# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and analyzing the FAIR^2 mass spectrometry human protein dataset using the `mlcroissant` library.

### Dataset Source
We utilize a Croissant schema published at:
- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Examine the available record sets, their `@id`s, and each field's details.


In [ ]:
# List all record sets and the fields and columns for context

record_sets = list(metadata.record_sets)
print(f"There are {len(record_sets)} record sets\n")
for i, rs in enumerate(record_sets):
    print(f"Record Set {i+1}:\n",
          f"  @id: {rs.id}\n",
          f"  name: {rs.name}")
    field_ids = [field.id for field in rs.fields]
    print(f"  Fields: {field_ids}")
    print("")

# Show detailed info for first record set's fields and columns
if record_sets:
    fields = list(record_sets[0].fields)
    for field in fields:
        print(f"Field @id: {field.id}\n  name: {field.name}\n  data_type: {field.data_type}")

## 3. Data Extraction
Load an entire record set into a DataFrame using the record set's `@id`. This enables analysis and EDA.

In [ ]:
# Load all record sets into dataframes by @id
dataframes = {}
for record_set in record_sets:
    records = list(dataset.records(record_set=record_set.id))
    df = pd.DataFrame(records)
    dataframes[record_set.id] = df

# Preview columns and first records for the first record set
main_rs_id = record_sets[0].id if record_sets else None
if main_rs_id:
    print(f"Columns for record set '{main_rs_id}':")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps on the main data record set.
- Filter records for a chosen numeric field (e.g., 'MW_Da' or similar field representing Molecular Weight in Daltons)
- Normalize numeric values
- Group data by a categorical field if present


In [ ]:
# Pick a numeric field based on record set fields overview
rs = record_sets[0]

# Try to find a suitable numeric field, e.g. 'MW_Da' (Molecular Weight in Daltons) or similar
field_candidates = [field for field in rs.fields if field.data_type in ('Float','Number','Integer') or 'weight' in (field.name or '').lower() or 'MW' in (field.name or '')]
if field_candidates:
    numeric_field_obj = field_candidates[0]
else:
    # fallback: just use the first field
    numeric_field_obj = list(rs.fields)[0]
numeric_field_id = numeric_field_obj.id

# Print the field chosen
print(f"Numeric field chosen for analysis: {numeric_field_obj.id}, name: {numeric_field_obj.name}\n")

df = dataframes[rs.id].copy()

# Remove missing/non-numeric rows
df = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()]
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Set a threshold for demonstration (e.g., MW > 50000)
threshold = df[numeric_field_id].quantile(0.5) if len(df) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Records with {numeric_field_id} > {threshold:.1f} (median value): {len(filtered_df)}\n")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field
categorical_candidates = [f for f in rs.fields if f.data_type=='Text' and f.id != numeric_field_id]
if categorical_candidates:
    group_field_id = categorical_candidates[0].id
    print(f'Grouping by: {group_field_id}')
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    display(grouped_df.head())

## 5. Visualization
Visualizing the distribution of the numeric field and relationship to the group field if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id], kde=True, color='dodgerblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# Boxplot grouped by categorical field if available
if categorical_candidates:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we used `mlcroissant` to examine the FAIR^2 human extracellular vesicle mass spectrometry dataset:
- Explored metadata and record set schemas using `@id` fields.
- Loaded all records for the primary record set to a DataFrame.
- Performed basic filtering, normalization, and grouped statistics on a major numeric field (`@id`: for example, molecular weight).
- Visualized key distributions and relationships between fields.

This foundation enables further analysis such as protein clustering, biomarker discovery, and advanced statistical modeling with clear provenance from Croissant.
